In [ ]:
# Import dependencies

from PIL import Image
import os
import json
from textwrap import dedent
import pandas as pd
import numpy as np
import random

import os
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_PI_Kersting/am84fuxo/LlavaGuard')

from llavaguard.taxonomy.PEGI.PEGI_Graph import policy_graph, get_overcat, policy_graph_to_safety_policy, get_max_pegi_score, get_content_categories_with_examples, graph_with_ratings, policy_graph_to_rating_policy
from llavaguard_config import workspace_dir, local_data_dir
from pegiguard.data.dataset_utils import parse_labels, labels_to_string


In [ ]:
# Parse category and subcategory labels

def parse_subcategories(value):
    """Parse lists and strings while excluding the canonical NA label."""
    return parse_labels(value)


In [ ]:
# Update rating prompts and answer fields

# Changing human_prompt and reordering GPT answer JSON to match the new human prompt structure.
import json
import shutil
from pathlib import Path

from llavaguard_config import local_data_dir
from llavaguard.taxonomy.PEGI.PEGI_Graph import policy_graph_to_rating_policy


MODE = "age_rating"
VERSION = "4"

file_path = Path(
    f"{local_data_dir}/datasets/PEGI-Guard-DS/v4/{MODE}/all_data.json"
)


print(f"Loading: {file_path}")

# Shutil.copy(file_path, backup_path).


with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)


def normalize_pegi_rating(value):
    """
    Converts:
        3 -> "3"
        "3" -> "3"
        "PEGI 3" -> "3"
    """
    if value is None:
        return None

    value = str(value).strip()
    value = value.replace("PEGI", "").strip()

    return value


def reorder_gpt_json(value):
    """
    Reorders GPT answer JSON to:
        rationale
        categories
        subcategories
        PEGI rating
    Also renames:
        pegi_rating -> PEGI rating
    """
    try:
        parsed = json.loads(value)
    except json.JSONDecodeError:
        print("WARNING: Could not parse GPT JSON:")
        print(value)
        return value

    new_parsed = {
        "rationale": parsed.get("rationale", ""),
        "categories": parsed.get("categories", "NA: None applying"),
        "subcategories": parsed.get("subcategories", "NA: None applying"),
        "PEGI rating": normalize_pegi_rating(
            parsed.get("PEGI rating", parsed.get("pegi_rating"))
        ),
    }

    return json.dumps(new_parsed, ensure_ascii=False, indent=4)


new_human_prompt = policy_graph_to_rating_policy()

for row in data:
    conversations = row.get("conversations", [])

    for conv in conversations:
        if conv.get("from") == "human":
            conv["value"] = new_human_prompt

        elif conv.get("from") == "gpt":
            conv["value"] = reorder_gpt_json(conv.get("value", ""))


with open(file_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Updated file saved to: {file_path}")


In [ ]:
# Serialize subcategory labels

# Converting subcategories from list to string for consistency with new human prompt.
import json
import shutil
from pathlib import Path

from llavaguard_config import local_data_dir


MODE = "age_rating"
VERSION = "7"

file_path = Path(
    f"{local_data_dir}/datasets/PEGI-Guard-DS/v4/{MODE}/all_data.json"
)

backup_path = file_path.with_suffix(".backup_subcategories.json")

shutil.copy(file_path, backup_path)

print(f"Backup saved to: {backup_path}")


def convert_subcategories_to_string(parsed):
    subcats = parsed.get("subcategories")

    if isinstance(subcats, list):
        parsed["subcategories"] = labels_to_string(subcats)

    return parsed


with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)


for sample in data:
    for conv in sample.get("conversations", []):

        if conv.get("from") != "gpt":
            continue

        try:
            parsed = json.loads(conv["value"])
        except Exception:
            print(f"Skipping malformed GPT output in {sample['id']}")
            continue

        parsed = convert_subcategories_to_string(parsed)

        conv["value"] = json.dumps(
            parsed,
            ensure_ascii=False,
            indent=4,
        )


with open(file_path, "w", encoding="utf-8") as f:
    json.dump(
        data,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Updated: {file_path}")


In [ ]:
# Correct UnsafeBench image paths

# Rename image paths in unsafebench to real file names.

input_csv  = f"{local_data_dir}/labels/pegi/v2/unsafebench.csv"

df = pd.read_csv(input_csv)
# 0 -> .../data/images/unsafebench/0.jpg.
for _, row in df.iterrows():
    idx = row['idx']
    img_path = row['image_path']
    if img_path is None:
        print(f"row {idx} has no image path")
        raise ValueError("image_path is None")
    new_path = f"{local_data_dir}/images/unsafebench/{idx}.jpg"
    df.at[_, 'image_path'] = new_path

output_file = f"{local_data_dir}/labels/pegi/v1/unsafebench_v2.csv"
df.to_csv(output_file, index=False)


In [ ]:
# Combine annotation sources

# Adding together the ds parts (llavaguard and unsafebench).
input_csv = f"{local_data_dir}/labels/pegi/v2/unsafebench.csv"

df = pd.read_csv(input_csv)

images = df['image_path'].unique()
categories = []
for img in images:
    df1 = df['image_path' == img]
    for _, row in df1:
        subcat = row['subcategory']
        assessment = row['assessment']
        if assessment == 'NO':
            continue
        
        categories.append(row['subcategory'])


In [ ]:
# Convert CSV annotations to JSON

def turn_csv_to_json(input_csv: str, output_json: str, checkpoint_every: int = 100, origin: str = "llavaguard"):
    """
    Read a CSV of image annotations, group rows by image_path,
    collect all accepted subcategories and their over-categories,
    and write the result to JSON.
    """

    df = pd.read_csv(input_csv)

    
    image_paths = df["image_path"].dropna().unique()

    results = []
   
    os.makedirs(os.path.dirname(output_json) or ".", exist_ok=True)

    total = len(image_paths)

    for idx, img_path in enumerate(image_paths, start=1):
        id = f"{origin}_{idx}"
        df_img = df[df["image_path"] == img_path]
        df_yes = df_img[df_img["assessment"] == "YES"].copy()
        subcategories = list(dict.fromkeys(df_yes["subcategory"].dropna()))
        
        over_categories = []
        for subcat in subcategories:
            oc = get_overcat(subcat)
            if oc not in over_categories:
                over_categories.append(oc)

        record = {
            "id": id,
            "image_path": img_path,
            "categories": over_categories,
            "subcategories": subcategories,
        }
        results.append(record)

        if (idx % checkpoint_every == 0) or (idx == total):
            with open(output_json, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=4)
            print(
                f"[Checkpoint] Wrote {idx}/{total} entries to {output_json}",
                flush=True,
            )

    print(f"\nDone. Wrote {len(results)} entries to: {output_json}")


In [ ]:
# Export selected annotations

input_csv = f"{local_data_dir}/labels/pegi/v1/labels_v3_no_educational_content_v1.csv"
output_json = f"{local_data_dir}/labels/pegi/v1/llavaguard_v2.json"
turn_csv_to_json(input_csv, output_json)


In [ ]:
# Combine source datasets

# Combine unsafebench json and llavaguard json.
import json

file_unsafebench = f"{local_data_dir}/labels/pegi/v1/unsafebench_v2.json"
file_llavaguard = f"{local_data_dir}/labels/pegi/v1/llavaguard_v2.json"
output_file = f"{local_data_dir}/labels/pegi/v1/pegiguard_merged.json"

with open(file_unsafebench, "r", encoding="utf-8") as f1:
    data1 = json.load(f1)
with open(file_llavaguard, "r", encoding="utf-8") as f2:
    data2 = json.load(f2)
merged = data1 + data2

with open(output_file, "w", encoding="utf-8") as f3:
    json.dump(merged, f3, ensure_ascii=False, indent=4)

print(f"Merged {len(data1)} + {len(data2)} = {len(merged)} records.")


In [ ]:
# Adjust legacy UnsafeBench IDs

import json
import os
import shutil

def fix_unsafebench_ids(input_file: str, output_file: str = None):
    """
    Adjusts all IDs in the given JSON dataset file:
    If an ID starts with 'unsafebench_' followed by a number (e.g. 'unsafebench_3238'),
    it becomes 'unsafebench_<number-1>'.
    All other IDs remain unchanged.

    Creates a backup of the original file if output_file == input_file.
    """
    # Read dataset.
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Fix IDs.
    prefix = "unsafebench_"
    for sample in data:
        sample_id = sample.get("id", "")
        if sample_id.startswith(prefix):
            try:
                number_part = int(sample_id[len(prefix):])
                sample["id"] = f"{prefix}{number_part - 1}"
            except ValueError:
                # Not a valid number after prefix, leave unchanged.
                continue

    # Determine output file.
    if output_file is None:
        output_file = input_file  # Overwrite in place.

    # Backup if overwriting.
    if output_file == input_file:
        backup_path = f"{input_file}.bak"
        shutil.copy(input_file, backup_path)
        print(f"Backup created at: {backup_path}")

    # Write corrected dataset.
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"Fixed IDs written to: {output_file}")


In [ ]:
# Regenerate age-dependent safety policies

import json
import os
import shutil

def fix_human_policies(input_file: str, output_file: str | None = None):
    """
    For each sample in the JSON dataset:
    - Read 'user age'
    - Recompute the human conversation with new policy_graph_to_safety_policy(age)
    - Replace conversations[i]['value'] where conversations[i]['from'] == 'human'
    Everything else is left unchanged.
    """

    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for sample in data:
        age = sample.get("user age", sample.get("user_age"))
        if age is None:
            continue

        fixed_policy = policy_graph_to_safety_policy(age)

        # Replace the human message in conversations.
        convs = sample.get("conversations", [])
        for msg in convs:
            if msg.get("from") == "human":
                msg["value"] = fixed_policy
                break

    if output_file is None:
        output_file = input_file 

    if output_file == input_file:
        backup_path = f"{input_file}.bak"
        shutil.copy(input_file, backup_path)
        print(f"Backup created at: {backup_path}")

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"Done. Fixed human policies written to: {output_file}")


In [ ]:
# Count subcategory annotations

import pandas as pd
import re

def count_subcategories(file_path: str):
    df = pd.read_json(file_path, orient="records")

    df["sub_list"] = df["subcategories"].apply(parse_subcategories)
    sub_df = df.explode("sub_list", ignore_index=True)
    sub_df = sub_df[sub_df["sub_list"].notna() & (sub_df["sub_list"] != "")]

    counts = sub_df["sub_list"].value_counts()

    # Divide numeric categories by 6 (age-duplicated samples).
    is_numeric = counts.index.to_series().str.match(r"^\d+")
    counts.loc[is_numeric] = counts.loc[is_numeric] / 6

    return counts


In [ ]:
# Serialize dataset label fields

def change_dataset_format():
    '''
    The function changes the "categories" and "subcategories" fields of the DS 
    from List to String to be compatible with llama-factory
    '''
    import json

    in_path  =f'{local_data_dir}/PEGI-Guard-DS/v1/fixed_x6_all_data.json'
    out_path = f'{local_data_dir}/PEGI-Guard-DS/v1/fixed_x6_all_data_string.json'

    with open(in_path, "r") as f:
        data = json.load(f)

    print("Total samples:", len(data))

    for ex in data:
        # Categories: force to string.
        cats = ex.get("categories")
        if isinstance(cats, list):
            ex["categories"] = labels_to_string(cats)
        elif cats is None:
            ex["categories"] = "NA: None applying"

        # Subcategories: same idea.
        sub = ex.get("subcategories")
        if isinstance(sub, list):
            ex["subcategories"] = labels_to_string(sub)
        elif sub is None:
            ex["subcategories"] = "NA: None applying"

    with open(out_path, "w") as f:
        json.dump(data, f, indent=2)

    print("Wrote cleaned file:", out_path)


In [ ]:
# Remove redundant subcategories

# Clean redundant categories from the dataset.

import pandas as pd
import json
from pathlib import Path


csv_path = f'{local_data_dir}/datasets/PEGI-Guard-DS/v4/all_data.csv'  # Use the cleaned of redundant categories version!!.
output_json_path = f'{local_data_dir}/datasets/PEGI-Guard-DS/v4/all_data.json'

df = pd.read_csv(csv_path)

subcat_to_cat = {}
for cat, subcats in policy_graph.items():
    for subcat in subcats.keys():
        subcat_to_cat[subcat] = cat

redundancy_chains = [
    ["9. Gross Violence", "6. Realistic Violence", "4. Mild Violence", "2. Unrealistic and Implied Violence"],
    ["12. Offensive Language", "11. Mild Swearing"],
    ["18. Explicit Sexual Content", "19. Explicit Nudity Content", "17. Mild Nudity Content", "16. Sexual Innuendo"],
    ["24. Highly Disturbing Content", "22. Moderately Disturbing Content", "21. Scary Content"],
    ["28. Suicide", "29. Self-injury", "27. Non-realistic Self-harm"]
]

def remove_redundant(subcategories):
    """Remove redundant subcategories based on hierarchy chains"""
    result = set(subcategories)
    for chain in redundancy_chains:
        # Find the highest severity present in this chain.
        for i, subcat in enumerate(chain):
            if subcat in result:
                # Remove all lower severity in this chain.
                for lower in chain[i+1:]:
                    result.discard(lower)
                break
    return result

final_dataset = []

for idx, (image_path, group) in enumerate(df.groupby('image_path')):
    # Get all YES subcategories for this image.
    yes_subcats = set(group[group['assessment'] == 'YES']['subcategory'].tolist())

    # Remove redundant subcategories.
    clean_subcats = remove_redundant(yes_subcats)

    # Get categories.
    categories = set()
    for subcat in clean_subcats:
        if subcat in subcat_to_cat:
            categories.add(subcat_to_cat[subcat])

    # Determine assessment.
    assessment = "Unsafe" if categories else "Safe"

    # Generate ID from image path.
    # Assuming path like /path/to/unsafebench/123.jpg -> unsafebench_123_0.
    path_parts = Path(image_path).parts
    
    if 'unsafebench' in path_parts:
        idx_part = path_parts.index('unsafebench')
        img_name = path_parts[idx_part+1] if idx_part+1 < len(path_parts) else 'unknown'
        img_num = img_name.replace('.jpg', '').replace('.png', '')
        img_id = f"unsafebench_{img_num}_0"
    else:
        img_id = f"llavaguard_{idx}"

    # Create record.
    record = {
        "id": img_id,
        "image": image_path,
        "categories": ", ".join(sorted(categories)) if categories else "NA: None applying",
        "subcategories": ", ".join(sorted(clean_subcats)) if clean_subcats else "NA: None applying",
        "user-rating": None,
        "assessment": None,
        "conversations": None  # To be generated later.
    }

    final_dataset.append(record)

# Save to JSON.
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(final_dataset, f, indent=2, ensure_ascii=False)

print(f"Dataset created with {len(final_dataset)} records")
print(f"Saved to: {output_json_path}")
print("\nSample record:")
print(json.dumps(final_dataset[0], indent=2) if final_dataset else "No records")


In [ ]:
# Load a dataset split

file_path = f"{local_data_dir}/datasets/PEGI-Guard-DS/v{VERSION}/{MODE}/{SET}.json"

print(f"Loading dataset from: {file_path}")

df = pd.read_json(file_path, orient="records")

print(f"Length of the df: {len(df)}")

# One row per image-subcategory pair.
df["sub_list"] = df["subcategories"].apply(parse_subcategories)
print(df["sub_list"].head()
      )


In [ ]:
# Configure plot colors

custom_colors = [
    "#488f31",
    "#5ea16d",
    "#9bb874",
    "#d7ce85",
    "#ce9c4c",
    "#c46428",
    "#b51022"
]
red_pallette = ["#B51022", "#e89084"]
green_pallette = ["#488f31", "#accda1"]
orange_pallette = ["#efd50d", "#d1570b"]
blue_pallette = ["#1f77b4", "#aec7e8"]
grey_pallette = ["#545252", "#c7c7c7"]


In [ ]:
# Plot subcategory distributions

import pandas as pd
import re
import seaborn as sns
import matplotlib.pyplot as plt

# Plot the ds distribution FOR SUBSET WITH STRING SUBCATS.
SET = "test"
MODE = "age_rating" # Or "age_rating".
VERSION = "6"


file_path = f"{local_data_dir}/datasets/PEGI-Guard-DS/v{VERSION}/{MODE}/{SET}.json"

print(f"Loading dataset from: {file_path}")

df = pd.read_json(file_path, orient="records")

print(f"Length of the df: {len(df)}")

# One row per image-subcategory pair.
df["categories"] = df["categories"].apply(parse_subcategories)

sub_df = df.explode("categories")

sub_counts = (
    sub_df.groupby("categories")["image path"]
    .nunique()
    .reset_index(name="count")
)

# Sort numerically by subcategory id.
sub_counts = sub_counts.sort_values(
    "categories",
    key=lambda s: s.str.extract(r"(\d+)")[0].fillna(999).astype(int)
)

plt.figure(figsize=(12, 16))

ax = sns.barplot(
    data=sub_counts,
    y="categories",
    x="count",
    palette=green_pallette
)
ax.set_title(f"Unique Images per Safety Subcategory in {SET} Set (v{VERSION}) of {MODE} Dataset")
ax.set_xlabel("Images Count")
ax.set_ylabel("Subcategory")
for container in ax.containers:
    ax.bar_label(container, fmt='%d')
plt.tight_layout()
plt.show()

df["pegi_rating"] = df["subcategories"].apply(highest_pegi_rating)

rating_order = [3, 7, 12, 16, 18]

plot_df = (
    df["pegi_rating"]
    .value_counts()
    .reindex(rating_order, fill_value=0)
    .rename_axis("PEGI Rating")
    .reset_index(name="Count")
)


plt.figure(figsize=(8, 5))

ax = sns.barplot(
    data=plot_df,
    x="PEGI Rating",
    y="Count",
    palette=green_pallette
)

ax.set_title(
    f"Image count by PEGI rating in {SET} set (v{VERSION}, {MODE})"
)

for container in ax.containers:
    ax.bar_label(container, fmt="%d")

plt.tight_layout()
plt.show()


In [ ]:
# Plot category distributions

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from llavaguard.taxonomy.PEGI.PEGI_Graph import get_rating
from llavaguard_config import local_data_dir


SET = "train"
MODE = "age_rating"
VERSION = "2"

PALLETTE = red_pallette

file_path = (f"{local_data_dir}/datasets/PEGI-Guard-DS/{MODE}/v{VERSION}/{SET}.json")

print(f"Loading dataset from: {file_path}")

df = pd.read_json(file_path, orient="records")

def highest_pegi_rating(subcategories):
    ratings = [
        get_rating(subcat)
        for subcat in subcategories
        if get_rating(subcat) is not None
    ]
    return max(ratings, default=3)

# Subcategory distribution.

sub_counts = (
    df.explode("subcategories")
        .groupby("subcategories")# ["image path"].
        # .nunique().
      .size()
      .reset_index(name="count")
      .sort_values(
          "subcategories",
          key=lambda s: s.str.extract(r"(\d+)")[0]
                          .fillna(999)
                          .astype(int)
      )
)

plt.figure(figsize=(12, 16))

ax = sns.barplot(
    data=sub_counts,
    y="subcategories",
    x="count",
    palette=PALLETTE
)

ax.set_title(
    f"Samples per Safety category in {SET} set (v{VERSION}, {MODE})"
)
ax.set_xlabel("Image Samples Count")
ax.set_ylabel("category")

for container in ax.containers:
    ax.bar_label(container, fmt="%d")

plt.tight_layout()
plt.show()

# PEGI distribution.

df["pegi_rating"] = df["subcategories"].apply(highest_pegi_rating)

rating_order = [3, 7, 12, 16, 18]

plot_df = (
    df["pegi_rating"]
    .value_counts()
    .reindex(rating_order, fill_value=0)
    .rename_axis("PEGI Rating")
    .reset_index(name="Count")
)


print(f"Total images: {plot_df['Count'].sum()}")

plt.figure(figsize=(8, 5))

ax = sns.barplot(
    data=plot_df,
    x="PEGI Rating",
    y="Count",
    palette=PALLETTE
)

ax.set_title(
    f"Sample count by PEGI rating in {SET} set (v{VERSION}, {MODE})"
)

for container in ax.containers:
    ax.bar_label(container, fmt="%d")

plt.tight_layout()
plt.show()


In [ ]:
# Parse safety categories

def parse_safety_categories(value):
    """Parse parent categories using the shared label parser."""
    return parse_labels(value)


In [ ]:
# Plot training-prefix distributions

import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from llavaguard_config import local_data_dir

SET = "train"
MODE = "age_rating"
VERSION = "2"

BREAKPOINTS = [1000, 5000, 7500, 10000, 11000]

red_pallette = ["#B51022", "#e89084"]
green_pallette = ["#488f31", "#accda1"]
orange_pallette = ["#d1570b", "#f9995e"]
blue_pallette = ["#1f77b4", "#aec7e8"]
purple_pallette = ["#6a3d9a", "#cab2d6"]
brown_pallette = ["#8b4513", "#deb887"]

BP_PALETTES = {
    1000: red_pallette,
    5000: green_pallette,
    7500: orange_pallette,
    10000: blue_pallette,
    11000: purple_pallette
    # 6: brown_pallette.
}

base_dir = f"{local_data_dir}/datasets/PEGI-Guard-DS/{MODE}/v{VERSION}"

plot_dir = f"{base_dir}/breakpoint_plots"
os.makedirs(plot_dir, exist_ok=True)

file_path = f"{base_dir}/{SET}.json"

print(f"Loading dataset from: {file_path}")

df = pd.read_json(file_path, orient="records")
df["subcategories"] = df["subcategories"].apply(parse_labels)


def parse_safety_categories(value):
    """Parse parent categories using the shared label parser."""
    return parse_labels(value)


def number_sort_key(series):
    return (
        series
        .astype(str)
        .str.extract(r"(\d+)")[0]
        .fillna(999)
        .astype(int)
    )


def safety_sort_key(series):
    return (
        series
        .astype(str)
        .str.extract(r"S(\d+)")[0]
        .fillna(999)
        .astype(int)
    )


def save_or_show(out_path):
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")


def plot_subcategory_distribution(bp_df, bp, palette):
    sub_counts = (
        bp_df.explode("subcategories")
        .groupby("subcategories")
        .size()
        .reset_index(name="count")
        .sort_values(
            "subcategories",
            key=number_sort_key,
        )
    )

    plt.figure(figsize=(12, 16))

    ax = sns.barplot(
        data=sub_counts,
        y="subcategories",
        x="count",
        palette=palette,
    )

    ax.set_title(
        f"Samples per subcategory in first {bp} train samples "
        f"(v{VERSION}, {MODE})"
    )
    ax.set_xlabel("Image sample count")
    ax.set_ylabel("Subcategory")

    for container in ax.containers:
        ax.bar_label(container, fmt="%d")

    out_path = f"{plot_dir}/subcategory_distribution_{bp}.png"
    save_or_show(out_path)

def plot_category_distribution(bp_df, bp, palette):
    bp_df = bp_df.copy()

    if "categories" not in bp_df.columns:
        print(f"Skipping category plot for bp={bp}: no 'category' column found.")
        return

    bp_df["parsed_categories"] = bp_df["categories"].apply(parse_safety_categories)

    cat_counts = (
        bp_df.explode("parsed_categories")
        .dropna(subset=["parsed_categories"])
        .groupby("parsed_categories")
        .size()
        .reset_index(name="count")
        .sort_values(
            "parsed_categories",
            key=safety_sort_key,
        )
    )

    plt.figure(figsize=(10, 6))

    ax = sns.barplot(
        data=cat_counts,
        y="parsed_categories",
        x="count",
        palette=palette,
    )

    ax.set_title(
        f"Samples per category in first {bp} train samples "
        f"(v{VERSION}, {MODE})"
    )
    ax.set_xlabel("Image sample count")
    ax.set_ylabel("Category")

    for container in ax.containers:
        ax.bar_label(container, fmt="%d")

    out_path = f"{plot_dir}/category_distribution_{bp}.png"
    save_or_show(out_path)


def plot_pegi_distribution(bp_df, bp, palette):
    bp_df = bp_df.copy()

    # Use the actual dataset label, not derived from subcategories.
    bp_df["pegi_rating"] = bp_df["age_rating"]

    rating_order = [3, 7, 12, 16, 18]

    plot_df = (
        bp_df["pegi_rating"]
        .value_counts()
        .reindex(rating_order, fill_value=0)
        .rename_axis("PEGI Rating")
        .reset_index(name="Count")
    )

    print(f"\nSample count per PEGI rating, first {bp} samples:")
    for _, row in plot_df.iterrows():
        print(f"PEGI {row['PEGI Rating']}: {row['Count']} images")
    print(f"Total images: {plot_df['Count'].sum()}")

    plt.figure(figsize=(8, 5))

    ax = sns.barplot(
        data=plot_df,
        x="PEGI Rating",
        y="Count",
        palette=palette,
    )

    ax.set_title(
        f"Sample count by PEGI rating in first {bp} train samples "
        f"(v{VERSION}, {MODE})"
    )
    ax.set_xlabel("PEGI rating")
    ax.set_ylabel("Image sample count")

    for container in ax.containers:
        ax.bar_label(container, fmt="%d")

    out_path = f"{plot_dir}/pegi_distribution_{bp}.png"
    save_or_show(out_path)


def plot_safety_category_distribution(bp_df, bp, palette):
    bp_df = bp_df.copy()

    if "category" not in bp_df.columns:
        print(f"Skipping safety-category plot for bp={bp}: no 'category' column found.")
        return

    bp_df["parsed_categories"] = bp_df["category"].apply(parse_safety_categories)

    cat_counts = (
        bp_df.explode("parsed_categories")
        .dropna(subset=["parsed_categories"])
        .groupby("parsed_categories")
        .size()
        .reset_index(name="count")
        .sort_values(
            "parsed_categories",
            key=safety_sort_key,
        )
    )

    plt.figure(figsize=(10, 6))

    ax = sns.barplot(
        data=cat_counts,
        y="parsed_categories",
        x="count",
        palette=palette,
    )

    ax.set_title(
        f"Samples per safety category in first {bp} train samples "
        f"(v{VERSION}, {MODE})"
    )
    ax.set_xlabel("Image sample count")
    ax.set_ylabel("Safety category")

    for container in ax.containers:
        ax.bar_label(container, fmt="%d")

    out_path = f"{plot_dir}/safety_category_distribution_{bp}.png"
    save_or_show(out_path)


for bp in BREAKPOINTS:
    print("=" * 100)
    print(f"Plotting distributions for first {bp} samples")
    print("=" * 100)

    if bp > len(df):
        print(f"Skipping breakpoint {bp}: dataset only has {len(df)} samples.")
        continue

    palette = BP_PALETTES[bp]
    bp_df = df.iloc[:bp].copy()

    plot_subcategory_distribution(bp_df, bp, palette)
    plot_category_distribution(bp_df, bp, palette)
    plot_pegi_distribution(bp_df, bp, palette)


In [ ]:
# Plot PEGI tag distributions

import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from llavaguard.taxonomy.PEGI.PEGI_Graph import get_pegi_tags

# Rotate this one to be like the above.


def plot_pegi_tags_distribution(file_path: str):
    """
    Plot the distribution of PEGI tags in the dataset.
    If an image has multiple subcategories with different tags, all tags are included.
    """
    df = pd.read_json(file_path, orient="records")
    
    def parse_subcategories(value):
        """Parse subcategories field into a list."""
        if isinstance(value, list):
            items = value
        elif isinstance(value, str):
            value = value.strip()
            if value in {"", "NA: None applying", "NA: None applying"}:
                return []
            items = re.split(r",\s*", value)
        else:
            return []
        return [item.strip() for item in items if item and item.strip() not in {"NA: None applying", "NA: None applying"}]
    
    def get_all_tags(subcategories):
        """Get all PEGI tags for a list of subcategories."""
        all_tags = []
        for subcat in parse_subcategories(subcategories):
            tags = get_pegi_tags(subcat)
            all_tags.extend(tags)
        return sorted(set(all_tags))
    
    # Parse and collect all tags.
    df["tags_list"] = df["subcategories"].apply(get_all_tags)
    
    # Explode: one row per tag.
    tags_df = df.explode("tags_list", ignore_index=True)
    
    # Filter out empty strings and None.
    tags_df = tags_df[tags_df["tags_list"].notna() & (tags_df["tags_list"] != "")]
    
    # Count tags.
    tag_counts = tags_df["tags_list"].value_counts().reset_index()
    tag_counts.columns = ["PEGI Tag", "Count"]
    
    # Plot.
    plt.figure(figsize=(12, 6))
    sns.barplot(data=tag_counts, x="Count", y="PEGI Tag", palette="magma")
    plt.title("Distribution of PEGI Tags in the Dataset")
    plt.xlabel("Count")
    plt.ylabel("PEGI Tag")
    
    for idx, value in enumerate(tag_counts["Count"]):
        plt.text(value + max(tag_counts["Count"]) * 0.01, idx, str(value), va="center")
    
    plt.tight_layout()
    plt.show()
    
    return tag_counts

# Usage.
file_path = f"{local_data_dir}/datasets/PEGI-Guard-DS/v4/all_data.json"
tag_distribution = plot_pegi_tags_distribution(file_path)
print(tag_distribution)
